In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(r'C:\Users\konta\Documents\DIV_Academy\Module2(From_29_nov)\data\housing.csv')

df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


In [2]:
outliers1 = df['median_house_value'] == df['median_house_value'].max()
outliers2 = df['housing_median_age'] == df['housing_median_age'].max()
outliers = outliers1 | outliers2

df = df.loc[~outliers, :].copy()

df.head(3)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
8,-122.26,37.84,42.0,2555.0,665.0,1206.0,595.0,2.0804,226700.0,NEAR BAY


In [3]:
X = df.drop('median_house_value', axis='columns')
y = df['median_house_value']

In [4]:
X['income_cat'] = pd.cut(
    X['median_income'],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    stratify=X['income_cat'],
    test_size=0.2
)

In [6]:
X_train.drop('income_cat', axis='columns', inplace=True)
X_test.drop('income_cat', axis='columns', inplace=True)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10):
        self.n_clusters = n_clusters

    def fit(self, X, y=None):
        self.kmeans_ = KMeans(self.n_clusters)
        self.kmeans_.fit(X)
        return self  

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=1).round(3)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

In [8]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

cat_cols = ['ocean_proximity']
log_cols = [
    'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income'
    ]

cat_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])
log_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p, inverse_func=np.expm1, feature_names_out='one-to-one')),
    ('scl', StandardScaler())
])

def ratio_func(arr_2d):
    return arr_2d[:, [0]] / arr_2d[:, [1]]
def ratio_feature_names(*args, **kwargs):
    return ['ratio']

rat_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('newcol', FunctionTransformer(ratio_func, feature_names_out=ratio_feature_names)),
    ('scl', StandardScaler())
])
cluster_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('cluster', ClusterSimilarity(n_clusters=10))
])
num_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scl', StandardScaler())
])

preprocessing = ColumnTransformer(
    transformers=[
        ('CAT', cat_pipeline, cat_cols),
        ('LOG', log_pipeline, log_cols),
        ('RAT_b/r', rat_pipeline, ['total_bedrooms', 'total_rooms']),
        ('RAT_r/h', rat_pipeline, ['total_rooms', 'households']),
        ('RAT_p/h', rat_pipeline, ['population', 'households']),
        ("GEO", cluster_pipeline, ["latitude", "longitude"]),
    ], remainder=num_pipeline
)

X_train_arr = preprocessing.fit_transform(X_train)
X_train_prepared = pd.DataFrame(X_train_arr, columns=preprocessing.get_feature_names_out())

X_test_arr = preprocessing.fit_transform(X_test)
X_test_prepared = pd.DataFrame(X_test_arr, columns=preprocessing.get_feature_names_out())

X_train_prepared

,CAT__ocean_proximity_<1H OCEAN,CAT__ocean_proximity_INLAND,CAT__ocean_proximity_ISLAND,CAT__ocean_proximity_NEAR BAY,CAT__ocean_proximity_NEAR OCEAN,LOG__total_rooms,LOG__total_bedrooms,LOG__population,LOG__households,LOG__median_income,...,GEO__Cluster 1 similarity,GEO__Cluster 2 similarity,GEO__Cluster 3 similarity,GEO__Cluster 4 similarity,GEO__Cluster 5 similarity,GEO__Cluster 6 similarity,GEO__Cluster 7 similarity,GEO__Cluster 8 similarity,GEO__Cluster 9 similarity,remainder__housing_median_age
0,0.0,1.0,0.0,0.0,0.0,0.624912,0.380297,0.565067,0.476370,0.371173,...,0.007,0.492,0.000,0.000,0.000,0.000,0.180,0.000,0.001,-1.576396
1,0.0,1.0,0.0,0.0,0.0,-1.232643,-1.662394,-1.843934,-1.540219,0.281123,...,0.000,0.008,0.001,0.324,0.000,0.878,0.000,0.000,0.000,0.000301
2,0.0,1.0,0.0,0.0,0.0,0.835256,0.957499,0.861125,0.972303,-0.312066,...,0.722,0.001,0.000,0.000,0.020,0.000,0.082,0.002,0.501,-1.313613
3,1.0,0.0,0.0,0.0,0.0,-0.651920,-0.621237,-0.310945,-0.605161,-1.477603,...,0.978,0.000,0.000,0.000,0.061,0.000,0.066,0.003,0.665,1.489403
4,0.0,1.0,0.0,0.0,0.0,0.074623,-0.576620,-0.275728,-0.463562,2.357378,...,0.701,0.000,0.000,0.000,0.144,0.000,0.011,0.027,0.949,-0.963236
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14852,0.0,1.0,0.0,0.0,0.0,-0.195366,-0.274790,0.133545,-0.057572,-1.423394,...,0.000,0.946,0.000,0.005,0.000,0.002,0.100,0.000,0.000,0.175489
14853,0.0,1.0,0.0,0.0,0.0,0.901818,0.509967,0.651185,0.609509,0.847834,...,0.000,0.010,0.001,0.403,0.000,0.799,0.000,0.000,0.000,-0.963236
14854,1.0,0.0,0.0,0.0,0.0,-0.056710,-0.026463,-0.290293,-0.011613,-0.473839,...,0.946,0.000,0.000,0.000,0.031,0.000,0.134,0.001,0.457,0.876243
14855,0.0,1.0,0.0,0.0,0.0,0.858270,0.543349,0.611463,0.466532,1.039942,...,0.000,0.001,0.008,0.086,0.000,0.943,0.000,0.000,0.000,-0.875642


In [9]:
from sklearn.linear_model import LinearRegression

lin_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("linreg", LinearRegression())
    ])
lin_reg.fit(X_train, y_train)
y_pred = lin_reg.predict(X_test)

In [10]:
y_pred

array([248019.68302108, 269583.23145592,  68719.79884595, ...,
       201992.37859536, 195767.41060784, 199538.34540079], shape=(3715,))

In [11]:
y_test.to_numpy()

array([186200., 253400.,  68100., ..., 263300., 166300., 387500.],
      shape=(3715,))

### Regression Metrics

In [15]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)
mae

41094.17224188563

In [16]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test, y_pred)
mse

3073650819.4612193

In [17]:
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(y_test, y_pred)
rmse

55440.51604613019

In [18]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)
r2

0.6602140205902616

### Adjusted $ R^2 $

In [19]:
def adjusted_r2_score(y_true, y_pred, n_features):
    n = len(y_true)
    p = n_features
    r2 = r2_score(y_true, y_pred)

    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

adj_r2 = adjusted_r2_score(y_test, y_pred, n_features=X.shape[1])
adj_r2

0.6592966718337558

### Mean Absolute Percentage Error (MAPE)

In [20]:
from sklearn.metrics import mean_absolute_percentage_error

mape = mean_absolute_percentage_error(y_test, y_pred)
mape

0.25478383863625204

### Huber Loss

In [21]:
def huber_loss_np(y_true, y_pred, delta=1.0):
    error = y_true - y_pred
    abs_error = np.abs(error)

    quadratic_loss = 0.5 * error ** 2
    linear_loss = delta * (abs_error - 0.5 * delta)

    loss = np.where(abs_error <= delta, quadratic_loss, linear_loss)

    return np.mean(loss)

huber_loss_np(y_test, y_pred, delta=1.0)

np.float64(41093.67224188563)